# 01 YOLO Segmentation Training Only for AlphaDent

This notebook is a **training-only** notebook for the AlphaDent YOLO segmentation workflow.

## Current experiment (V10)

This is the **V10** experiment: `yolov8s-seg + imgsz=768 + mild rare Caries oversampling (x2) + default augmentation`.

Changes from V9 (`yolov8m-seg + x3 oversampling + reduced mosaic`):

1. Switch model back to `yolov8s-seg.pt` (the best-known model from V6).
2. Reduce oversampling repeat factor from `3` to `2` (mild: duplicate once only).
3. Restore mosaic augmentation to YOLO default (`1.0`).
4. Restore batch size to `16` (safe for `yolov8s-seg` at `imgsz=768`, confirmed in V6).

The goal of V10 is to test whether **mild rare Caries oversampling** improves performance without causing the precision drop seen in V7/V9.

Other settings (`epochs`, `patience`, `fliplr`, `workers`, logging callbacks, output structure) are unchanged.

The intended final training outputs are:

- `weights/best.pt`  ← main checkpoint for validation/submission
- `weights/last.pt`  ← only for comparison/debugging
- `results.csv`

A separate notebook should be used later for model validation, prediction visualization, error analysis, and submission preparation.

Use `weights/best.pt` for validation and submission. Do **not** judge the run by `weights/last.pt`.

## 1. Environment Setup

This section imports the basic packages and sets a fixed random seed for reproducibility.

In [ ]:
import os
import random
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Current working directory:", os.getcwd())
print("Kaggle input directory exists:", Path("/kaggle/input").exists())
print("Kaggle working directory exists:", Path("/kaggle/working").exists())

In [ ]:
# Check GPU status.
# This is useful on Kaggle to confirm that the selected accelerator is available.
!nvidia-smi

## 2. Install and Import Ultralytics

If Ultralytics is already installed, this cell will finish quickly. If Internet is disabled, install Ultralytics from an uploaded Kaggle Dataset containing the required wheels.

In [ ]:
# Install Ultralytics if it is not already available.
try:
    import ultralytics
    print("Ultralytics is already installed.")
except ImportError:
    print("Installing Ultralytics...")
    !pip install -q ultralytics

In [ ]:
# Disable most progress-bar style logs before importing YOLO.
# This helps keep Kaggle notebook logs cleaner.
os.environ["YOLO_VERBOSE"] = "False"

from ultralytics import YOLO
from ultralytics.utils import LOGGER
import ultralytics

# Keep only important Ultralytics logs.
LOGGER.setLevel(logging.ERROR)

# Extra safeguard against verbose progress bars in some Ultralytics versions.
try:
    import ultralytics.utils as yolo_utils
    yolo_utils.VERBOSE = False
except Exception:
    pass

try:
    import ultralytics.utils.tqdm as yolo_tqdm
    yolo_tqdm.VERBOSE = False
except Exception:
    pass

print("Ultralytics version:", ultralytics.__version__)

## 3. Locate the Dataset YAML

This notebook searches for `yolo_seg_train.yaml` under `/kaggle/input` and uses it as the YOLO dataset configuration file.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

yaml_candidates = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))

print(f"Number of yolo_seg_train.yaml files found: {len(yaml_candidates)}")

for i, p in enumerate(yaml_candidates):
    print(f"[{i}] {p}")

if len(yaml_candidates) == 0:
    raise FileNotFoundError(
        "No yolo_seg_train.yaml found under /kaggle/input. "
        "Please check whether the AlphaDent dataset is added to this notebook."
    )

# If multiple YAML files are found, use the first one by default.
# Change YAML_INDEX manually if needed.
YAML_INDEX = 0
DATA_YAML = yaml_candidates[YAML_INDEX]

print("\nSelected DATA_YAML:", DATA_YAML)
print("Dataset root:", DATA_YAML.parent)

## 4. Inspect Dataset Metadata

This section reads the YAML file and checks the class names.

In [ ]:
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_yaml_dict = yaml.safe_load(f)

print("YAML content:")
for k, v in data_yaml_dict.items():
    print(f"{k}: {v}")

print("\nClass names:")
names = data_yaml_dict.get("names", None)
print(names)

if names is None:
    raise ValueError("The YAML file does not contain the 'names' field.")

if isinstance(names, dict):
    num_classes = len(names)
elif isinstance(names, list):
    num_classes = len(names)
else:
    raise TypeError("The 'names' field should be either a list or a dictionary.")

print("Number of classes:", num_classes)

## 5. Basic Path Check

This section resolves the train and validation image folders. The resolved paths are used to create a runtime YAML file with absolute paths for Ultralytics.

In [ ]:
dataset_root = DATA_YAML.parent

yaml_path_value = data_yaml_dict.get("path", None)

if yaml_path_value is not None:
    yaml_root = Path(yaml_path_value)
    if not yaml_root.is_absolute():
        yaml_root = dataset_root / yaml_root
else:
    yaml_root = dataset_root

print("Resolved YAML root:", yaml_root)

def resolve_split_path(split_value):
    # Resolve a split path from the YAML file.
    if split_value is None:
        return None

    split_path = Path(split_value)
    if split_path.is_absolute():
        return split_path

    # Try resolving relative to YAML root first.
    candidate_1 = yaml_root / split_path
    if candidate_1.exists():
        return candidate_1

    # Try resolving relative to the YAML file directory.
    candidate_2 = dataset_root / split_path
    if candidate_2.exists():
        return candidate_2

    return candidate_1

train_path = resolve_split_path(data_yaml_dict.get("train", None))
val_path = resolve_split_path(data_yaml_dict.get("val", data_yaml_dict.get("valid", None)))
test_path = resolve_split_path(data_yaml_dict.get("test", None))

print("Train path:", train_path, "| exists:", train_path.exists() if train_path else None)
print("Val path:", val_path, "| exists:", val_path.exists() if val_path else None)
print("Test path:", test_path, "| exists:", test_path.exists() if test_path else None)

if train_path is None or not train_path.exists():
    raise FileNotFoundError("Train path does not exist. Please check the YAML file.")

if val_path is None or not val_path.exists():
    raise FileNotFoundError("Validation path does not exist. Please check the YAML file.")

In [ ]:
# Count image files in each split.
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder):
    if folder is None or not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in image_exts)

split_counts = {
    "train": count_images(train_path),
    "val": count_images(val_path),
    "test": count_images(test_path),
}

pd.DataFrame(
    [{"split": k, "num_images": v} for k, v in split_counts.items()]
)

## 6. Training Configuration

Change the parameters in this section for different experiments. The `RUN_NAME` is generated automatically from the key training parameters so that the result folder name records the experiment setup.

In [ ]:
# =========================
# Training configuration
# =========================

# V10: switch back to YOLOv8s-seg, which was the best model in V6.
# yolov8m-seg (V9) did not improve validation performance over yolov8s-seg (V6).
MODEL_NAME = "yolov8s-seg.pt"

# Keep the same maximum training budget.
# The final model should be selected from weights/best.pt, not weights/last.pt.
EPOCHS = 120

# Set image size back to 768.
# This keeps the best previous image size while testing the larger YOLOv8m-seg model.
IMG_SIZE = 768

# yolov8s-seg at imgsz=768 fits comfortably with batch=16 on Kaggle T4 (confirmed in V6).
BATCH_SIZE = 16

DEVICE = 0
WORKERS = 2

# Change 2: keep early stopping and rely on best.pt.
# A slightly larger patience is used so that the higher-resolution run has enough time,
# but best.pt remains the checkpoint used for later validation/submission.
PATIENCE = 25

# AlphaDent classes are pathology types, not left/right positions.
# Keeping horizontal flip disabled avoids creating unrealistic dental layouts.
FLIPLR = 0.0

# V10: restore YOLO default mosaic augmentation.
# Reduced mosaic was part of V7/V8/V9 strategy; V10 tests baseline augmentation.
MOSAIC = 1.0
CLOSE_MOSAIC = 20
MIXUP = 0.0
COPY_PASTE = 0.0

# Change 4: oversample images that contain rare Caries classes.
# The rare classes are inferred from class names containing "Caries 3", "Caries 4",
# "Caries 5", or "Caries 6". These classes were weak in the previous error analysis.
USE_RARE_CARIES_OVERSAMPLING = True
RARE_CARIES_KEYWORDS = ["caries 3", "caries 4", "caries 5", "caries 6"]

# Total number of times a rare-Caries image appears in the generated training list.
# 2 means: original image + 1 duplicate (mild oversampling, as recommended for V10).
RARE_CARIES_REPEAT_FACTOR = 2

PROJECT_DIR = Path("/kaggle/working/alphadent_yolo")

# Automatically include key parameters in the run folder name.
model_tag = Path(MODEL_NAME).stem.replace("-", "_")
fliplr_tag = str(FLIPLR).replace(".", "p")
mosaic_tag = str(MOSAIC).replace(".", "p")
oversample_tag = f"rarecariesx{RARE_CARIES_REPEAT_FACTOR}" if USE_RARE_CARIES_OVERSAMPLING else "nooversample"

RUN_NAME = (
    f"{model_tag}_img{IMG_SIZE}_bs{BATCH_SIZE}_"
    f"ep{EPOCHS}_pat{PATIENCE}_"
    f"mosaic{mosaic_tag}_close{CLOSE_MOSAIC}_"
    f"{oversample_tag}_fliplr{fliplr_tag}_seed{SEED}"
)

print("Model:", MODEL_NAME)
print("Epochs:", EPOCHS)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Patience:", PATIENCE)
print("fliplr:", FLIPLR)
print("mosaic:", MOSAIC)
print("close_mosaic:", CLOSE_MOSAIC)
print("mixup:", MIXUP)
print("copy_paste:", COPY_PASTE)
print("Use rare Caries oversampling:", USE_RARE_CARIES_OVERSAMPLING)
print("Rare Caries repeat factor:", RARE_CARIES_REPEAT_FACTOR)
print("Project directory:", PROJECT_DIR)
print("Run name:", RUN_NAME)

## 7. Rare Caries Oversampling

This section creates a training image list with duplicate entries for images that contain rare Caries classes.

This does **not** modify the original images or labels.  
It only writes a new text file under `/kaggle/working` and points YOLO training to that list.

The goal is to expose the model more often to rare and very small Caries targets.

In [ ]:
# =========================
# Rare Caries oversampling
# =========================

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def normalize_names(names_obj):
    # Return class names as a list ordered by class id.
    if isinstance(names_obj, dict):
        return [str(names_obj[k]) for k in sorted(names_obj.keys(), key=lambda x: int(x))]
    if isinstance(names_obj, list):
        return [str(x) for x in names_obj]
    raise TypeError("The YAML 'names' field must be either a list or a dictionary.")

def list_images_from_split(split_path):
    # List images from a YOLO split path. The split can be a folder or a txt file.
    split_path = Path(split_path)

    if split_path.is_file():
        image_paths = []
        for line in split_path.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            p = Path(line)
            if not p.is_absolute():
                p = yaml_root / p
            if p.suffix.lower() in IMAGE_EXTS:
                image_paths.append(p)
        return sorted(image_paths)

    if split_path.is_dir():
        return sorted([p for p in split_path.rglob("*") if p.suffix.lower() in IMAGE_EXTS])

    raise FileNotFoundError(f"Training split path does not exist: {split_path}")

def image_to_label_path(image_path):
    # Map a YOLO image path to the corresponding label txt path.
    image_path = Path(image_path)
    parts = list(image_path.parts)

    # Common YOLO structure:
    # .../images/train/xxx.jpg -> .../labels/train/xxx.txt
    if "images" in parts:
        idx = len(parts) - 1 - parts[::-1].index("images")
        parts[idx] = "labels"
        return Path(*parts).with_suffix(".txt")

    # Fallback structure:
    # parent/images -> parent/labels
    return image_path.parent.parent / "labels" / image_path.parent.name / f"{image_path.stem}.txt"

def read_label_class_ids(label_path):
    # Read YOLO class ids from a label file.
    label_path = Path(label_path)
    if not label_path.exists():
        return []

    class_ids = []
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 0:
            continue
        try:
            class_ids.append(int(float(parts[0])))
        except ValueError:
            continue
    return class_ids

class_names = normalize_names(names)

rare_caries_class_ids = [
    idx for idx, class_name in enumerate(class_names)
    if any(keyword in class_name.lower() for keyword in RARE_CARIES_KEYWORDS)
]

print("Detected class names:")
for idx, class_name in enumerate(class_names):
    print(f"{idx}: {class_name}")

print("\nRare Caries keywords:", RARE_CARIES_KEYWORDS)
print("Detected rare Caries class ids:", rare_caries_class_ids)

train_images = list_images_from_split(train_path)
print("\nNumber of original training images:", len(train_images))

if len(train_images) == 0:
    raise RuntimeError("No training images were found. Please check train_path and the dataset YAML.")

oversampled_train_list = []
image_level_records = []

for image_path in train_images:
    label_path = image_to_label_path(image_path)
    class_ids = read_label_class_ids(label_path)
    has_rare_caries = any(class_id in rare_caries_class_ids for class_id in class_ids)

    repeat = RARE_CARIES_REPEAT_FACTOR if (USE_RARE_CARIES_OVERSAMPLING and has_rare_caries) else 1

    for _ in range(repeat):
        oversampled_train_list.append(str(image_path))

    image_level_records.append({
        "image_path": str(image_path),
        "label_path": str(label_path),
        "n_objects": len(class_ids),
        "class_ids": class_ids,
        "has_rare_caries": has_rare_caries,
        "repeat": repeat,
    })

oversample_df = pd.DataFrame(image_level_records)

print("Images containing rare Caries classes:", int(oversample_df["has_rare_caries"].sum()))
print("Original training list length:", len(train_images))
print("Oversampled training list length:", len(oversampled_train_list))

if len(rare_caries_class_ids) == 0:
    print("\nWarning: No rare Caries classes were detected from class names.")
    print("The training list will not be meaningfully oversampled unless class names match the configured keywords.")

print("\nRepeat factor summary:")
display(oversample_df["repeat"].value_counts().sort_index().to_frame("n_images"))

OVERSAMPLED_TRAIN_TXT = Path("/kaggle/working/train_rare_caries_oversampled.txt")

with open(OVERSAMPLED_TRAIN_TXT, "w", encoding="utf-8") as f:
    for image_path in oversampled_train_list:
        f.write(f"{image_path}\n")

# This variable is used by the runtime YOLO YAML creation cell.
if USE_RARE_CARIES_OVERSAMPLING:
    TRAIN_SOURCE_FOR_YOLO = OVERSAMPLED_TRAIN_TXT
else:
    TRAIN_SOURCE_FOR_YOLO = train_path

print("\nTraining source used by YOLO:")
print(TRAIN_SOURCE_FOR_YOLO)
print("Exists:", Path(TRAIN_SOURCE_FOR_YOLO).exists())

## 8. Create Runtime YOLO YAML

This cell writes a stable runtime YAML file for training.

If rare Caries oversampling is enabled, the `train` field points to the generated oversampled training text file.  
The validation path is kept unchanged.

In [ ]:
YOLO_DATA_YAML = Path("/kaggle/working/yolo_seg_train_runtime.yaml")

def make_relative_or_absolute(path_value, root_value):
    path_value = Path(path_value)
    root_value = Path(root_value)
    try:
        return str(path_value.relative_to(root_value))
    except ValueError:
        return str(path_value)

# Use the oversampled train list if it was created.
train_source = Path(TRAIN_SOURCE_FOR_YOLO) if "TRAIN_SOURCE_FOR_YOLO" in globals() else train_path

fixed_yaml = dict(data_yaml_dict)
fixed_yaml["path"] = str(DATA_YAML.parent)
fixed_yaml["train"] = str(train_source) if train_source.is_file() else make_relative_or_absolute(train_source, DATA_YAML.parent)
fixed_yaml["val"] = make_relative_or_absolute(val_path, DATA_YAML.parent)
fixed_yaml["names"] = names

if test_path is not None and Path(test_path).exists():
    fixed_yaml["test"] = make_relative_or_absolute(test_path, DATA_YAML.parent)

with open(YOLO_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(fixed_yaml, f, sort_keys=False)

print("Runtime YOLO YAML:", YOLO_DATA_YAML)
print("Runtime YOLO YAML content:")
with open(YOLO_DATA_YAML, "r", encoding="utf-8") as f:
    print(f.read())

## 9. Train YOLO Segmentation Model

This training run uses the same training pipeline as the uploaded notebook, with only these necessary changes:

- model: `yolov8m-seg.pt`
- image size: `imgsz=768`

Other settings are intentionally kept unchanged from the uploaded notebook, including rare Caries oversampling and augmentation settings.

For later validation or submission, use `weights/best.pt` instead of `weights/last.pt`.

In [ ]:
model = YOLO(MODEL_NAME)

# Custom concise logging callbacks.
# These callbacks print one message at the start of each epoch and one summary at the end.
def on_train_epoch_start(trainer):
    current_epoch = trainer.epoch + 1
    total_epochs = trainer.epochs
    print(f"\n========== Epoch {current_epoch}/{total_epochs} started ==========")


def on_fit_epoch_end(trainer):
    current_epoch = trainer.epoch + 1
    total_epochs = trainer.epochs

    print(f"---------- Epoch {current_epoch}/{total_epochs} finished ----------")

    # Print training losses if available.
    if hasattr(trainer, "tloss") and trainer.tloss is not None:
        try:
            loss_values = trainer.tloss.detach().cpu().numpy().tolist()
            loss_names = getattr(trainer, "loss_names", None)

            if loss_names is not None and len(loss_names) == len(loss_values):
                loss_text = ", ".join(
                    [f"{name}: {value:.4f}" for name, value in zip(loss_names, loss_values)]
                )
            else:
                loss_text = ", ".join(
                    [f"loss_{i}: {value:.4f}" for i, value in enumerate(loss_values)]
                )

            print("Train losses:", loss_text)
        except Exception as e:
            print("Train losses could not be printed:", e)

    # Print the main validation metrics recorded during training.
    if hasattr(trainer, "metrics") and trainer.metrics is not None:
        metrics_to_show = [
            "metrics/precision(B)",
            "metrics/recall(B)",
            "metrics/mAP50(B)",
            "metrics/mAP50-95(B)",
            "metrics/precision(M)",
            "metrics/recall(M)",
            "metrics/mAP50(M)",
            "metrics/mAP50-95(M)",
        ]

        metric_text_list = []
        for key in metrics_to_show:
            if key in trainer.metrics:
                metric_text_list.append(f"{key}: {trainer.metrics[key]:.4f}")

        if len(metric_text_list) > 0:
            print("Validation metrics:")
            print(", ".join(metric_text_list))


model.add_callback("on_train_epoch_start", on_train_epoch_start)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

results = model.train(
    data=str(YOLO_DATA_YAML),
    task="segment",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    seed=SEED,
    patience=PATIENCE,
    cache=False,
    pretrained=True,
    optimizer="auto",
    verbose=False,
    plots=False,
    fliplr=FLIPLR,
    mosaic=MOSAIC,
    close_mosaic=CLOSE_MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
)

# Store the actual output directory created by Ultralytics.
RUN_DIR = Path(model.trainer.save_dir)
WEIGHTS_DIR = RUN_DIR / "weights"
BEST_PT = WEIGHTS_DIR / "best.pt"
LAST_PT = WEIGHTS_DIR / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

print("\nActual Ultralytics run directory:", RUN_DIR)
print("Weights directory:", WEIGHTS_DIR)
print("best.pt exists:", BEST_PT.exists())
print("last.pt exists:", LAST_PT.exists())
print("results.csv exists:", RESULTS_CSV.exists())

print("\nImportant checkpoint note:")
print("Use best.pt for validation/submission. last.pt is kept only for comparison/debugging.")

## 10. Check Training Outputs

This cell checks that the required outputs were created.

The main checkpoint for later analysis and submission is:

- `weights/best.pt`

`weights/last.pt` is still saved, but it should not be used unless you explicitly want to compare it against `best.pt`.

In [ ]:
# Use the actual output directory from training if available.
# If the kernel was restarted, search for the latest matching run folder.

if "RUN_DIR" in globals() and Path(RUN_DIR).exists():
    RUN_DIR = Path(RUN_DIR)
else:
    candidate_run_dirs = sorted(
        [p for p in PROJECT_DIR.glob(f"{RUN_NAME}*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    print("Detected candidate run directories:")
    for p in candidate_run_dirs:
        print(p)

    RUN_DIR = None
    for p in candidate_run_dirs:
        if (p / "weights").exists():
            RUN_DIR = p
            break

    if RUN_DIR is None:
        raise FileNotFoundError("No YOLO run directory with a weights folder was found.")

WEIGHTS_DIR = RUN_DIR / "weights"
BEST_PT = WEIGHTS_DIR / "best.pt"
LAST_PT = WEIGHTS_DIR / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

required_outputs = {
    "best.pt": BEST_PT,
    "last.pt": LAST_PT,
    "results.csv": RESULTS_CSV,
}

print("Run directory:", RUN_DIR)
print("\nRequired training outputs:")

missing = []
for name, path in required_outputs.items():
    exists = path.exists()
    print(f"{name}: {path} | exists: {exists}")
    if not exists:
        missing.append(name)

if missing:
    raise FileNotFoundError(f"Missing required output files: {missing}")

print("\nTraining-only notebook completed successfully.")
print("Main checkpoint for the next notebook:", BEST_PT)
print("Use best.pt for error analysis and submission preparation.")

In [ ]:
# Optional: show only the last few rows of the training metrics CSV.
# This does not generate plots or image outputs.

metrics_df = pd.read_csv(RESULTS_CSV)
metrics_df.columns = [c.strip() for c in metrics_df.columns]
metrics_df.tail()

## Next Step

Run the separate error analysis notebook to compare this run against the previous `yolov8s-seg + imgsz=768` baseline.

The main question is:

> Does switching from `yolov8s-seg` to `yolov8m-seg` improve validation performance when the image size is kept at `768`?

Focus on:

1. `Mask mAP50-95`
2. `Mask mAP50`
3. `Mask Precision`
4. `Mask Recall`
5. per-class performance, especially the Caries classes
6. whether false positives increase or decrease

Do not judge this run by `last.pt`; use `best.pt`.